In [ ]:
import json

import numpy as np
import matplotlib.pyplot as plt
from sklearn.metrics import roc_auc_score, roc_curve

from qgravnet import GravNetFactory

from hls4ml_gravnet.utils.data import load_processed, shuffle_vertices
from hls4ml_gravnet.utils.evaluation import load_run, response_rmse
from hls4ml_gravnet.utils.files import get_project_root_dir

PROJECT_ROOT = get_project_root_dir("hls4ml-gravnet")

### Load Model & Keras Predict

In [ ]:
RUN = "mini_128_L1_FP"
model_cfg, weights_path, history, datapath, n_vertices, is_shuffled = load_run(PROJECT_ROOT / "data/results/" / RUN)
info = json.load(open(PROJECT_ROOT / "data/results/" / RUN / "info.json"))
assert info["factory"] == "GravNetFactory", f"Expected factory to be GravNetFactory, but got {info['factory']}."
D = load_processed(datapath)
trained_model = GravNetFactory(**model_cfg).create_keras_model(n_vertices, 4)
trained_model.load_weights(weights_path)

if is_shuffled:
    D["X_hits_test"] = shuffle_vertices(D["X_hits_test"], seed=0)
X_test = np.copy(D["X_hits_test"][:, :n_vertices, :])
test_energy_pred, test_pid_pred = trained_model.predict(X_test)

### Keras Eval

In [ ]:
test_response_rmse = response_rmse(D["y_energy_test"], test_energy_pred)
test_auc = roc_auc_score(D["y_pid_test"], test_pid_pred)
fpr, tpr, thresholds = roc_curve(D["y_pid_test"], test_pid_pred)

print(f"Test Response RMSE: {test_response_rmse:.4f}")
print(f"Test PID AUC: {test_auc:.4f}")

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
plt.plot(fpr, tpr)
plt.xlabel("Pion False Positive Rate")
plt.ylabel("Pion True Positive Rate")
plt.xlim(0.0, 1.0)
plt.ylim(0.0, 1.0)

plt.subplot(1, 2, 2)
plt.hist(
    test_energy_pred.flatten() / D["y_energy_test"],
    bins=50,
    alpha=0.7,
    density=True,
)
plt.axvline(1.0, color="k", linestyle="--", lw=1, alpha=0.7)
plt.xlabel("Predicted / True Energy")
plt.ylabel("Density")
plt.xlim(0.0, 4.0)

spcr = " " * 5
notes_dataset = "Trained on small Garnet dataset \n(1 file, 10k events)" if "mini" in datapath else "Trained on full Garnet dataset \n(50 files, à 10k events)"
notes = f"{notes_dataset}\n\n{n_vertices} vertices\n\nNo large skip connections"
descr = (
    "QGravNet Evaluation \n\n"
    + spcr
    + f"AUC: {test_auc:.3f} \n"
    + spcr
    + f"Response RMS: {test_response_rmse:.3f}"
    + "\n\nNotes: \n\n"
    + notes
)
plt.text(
    1.05,
    1.0,
    descr,
    transform=plt.gca().transAxes,
    fontsize=11,
    verticalalignment="top",
    horizontalalignment="left",
)

plt.tight_layout()

plt.savefig(f"{PROJECT_ROOT}/data/results/{RUN}/eval_plot.png", dpi=300)
plt.savefig(f"{PROJECT_ROOT}/data/results/{RUN}/eval_plot.pdf")
plt.show()

### Export, Convert to ONNX and test in ONNX runtime

In [ ]:
export_path = PROJECT_ROOT / "data/results/" / RUN / "export"
export_keras, export_onnx = str(export_path / "keras"), str(export_path / "model.onnx")
trained_model.export(export_keras)

keras_reg, keras_cls = test_energy_pred, test_pid_pred
np.save(export_path / "X_test.npy", X_test.astype(np.float32))
np.savez_compressed(export_path / "keras_outputs.npz", reg=keras_reg, cls=keras_cls)

In [ ]:
X_test.shape

In [ ]:
!python -m tf2onnx.convert --saved-model {export_keras} --output {export_onnx} --opset 17

In [ ]:
import onnxruntime as ort

session = ort.InferenceSession(export_onnx)

input_name = session.get_inputs()[0].name
onnx_out = session.run(None, {input_name: X_test.astype(np.float32)})

In [ ]:
keras_response = keras_reg.flatten() / D["y_energy_test"]

onnx_reg, onnx_cls = onnx_out
onnx_response = onnx_reg.flatten() / D["y_energy_test"]
onnx_roc = roc_curve(D["y_pid_test"], onnx_cls)

keras_auc, onnx_auc = roc_auc_score(D["y_pid_test"], keras_cls), roc_auc_score(D["y_pid_test"], onnx_cls)
keras_r_rmse, onnx_r_rmse = response_rmse(D["y_energy_test"], keras_reg.flatten()), response_rmse(D["y_energy_test"], onnx_reg.flatten())

reg_delta = onnx_reg.flatten() - keras_reg.flatten()
cls_delta = onnx_cls.flatten() - keras_cls.flatten()

prec = 2
print_delta = lambda delta: f"Abs mean={abs(delta).mean():.{prec}e}\nAbs max={abs(delta).max():.{prec}e}\n\nAbs rel mean={abs(delta / keras_reg.flatten()).mean():.{prec}e}\nAbs rel max={abs(delta / keras_reg.flatten()).max():.{prec}e}"

print("-"*10, "ONNX vs Keras Comparison", "-"*10)
print(f"Absolute Regression Delta: mean={abs(reg_delta).mean():.4e}, max={abs(reg_delta).max():.4e}")
print(f"Absolute Classification Delta: mean={abs(cls_delta).mean():.4e}, max={abs(cls_delta).max():.4e}")
print(f"\nONNX AUC: {onnx_auc:.4f} vs Keras AUC: {keras_auc:.4f}")
print(f"ONNX Response RMSE: {onnx_r_rmse:.4f} vs Keras Response RMSE: {keras_r_rmse:.4f}")

plt.figure(figsize=(9, 4))

plt.subplot(1, 2, 1)
_, bins, _ = plt.hist(reg_delta, bins=75, density=False)
plt.xlabel("Regression Output Delta")
plt.ylabel("Counts")
plt.yscale("log")
plt.title("ONNX vs Keras Regression Output Difference")
plt.text(0.95, 0.95, print_delta(reg_delta), transform=plt.gca().transAxes, fontsize=10, verticalalignment="top", horizontalalignment="right")

plt.subplot(1, 2, 2)
_, bins, _ = plt.hist(cls_delta, bins=75, density=False)
plt.xlabel("Classification Output Delta")
plt.ylabel("Counts")
plt.yscale("log")
plt.title("ONNX vs Keras Classification Output Difference")
plt.text(0.05, 0.95, print_delta(cls_delta), transform=plt.gca().transAxes, fontsize=10, verticalalignment="top", horizontalalignment="left")

plt.tight_layout()
plt.savefig(f"{PROJECT_ROOT}/data/results/{RUN}/export/onnx_keras_delta_comparison.png", dpi=300)
plt.savefig(f"{PROJECT_ROOT}/data/results/{RUN}/export/onnx_keras_delta_comparison.pdf")
plt.show()

# Compare ONNX outputs to Keras outputs
lw = 1.5
fig = plt.figure(figsize=(9, 4))
plt.subplot(1, 2, 1)
plt.plot(fpr, tpr, label="Keras", lw=lw)
plt.plot(*onnx_roc[:2], label="ONNX", linestyle="--", lw=lw)
plt.xlabel("Pion False Positive Rate")
plt.ylabel("Pion True Positive Rate")
plt.xlim(0.0, 1.0)
plt.ylim(0.0, 1.0)
plt.title("ROC Curve Comparison")
plt.text(0.95, 0.95, f"Keras AUC: {keras_auc:.4f}\nONNX AUC: {onnx_auc:.4f}", transform=plt.gca().transAxes, fontsize=10, verticalalignment="top", horizontalalignment="right")
handles, labels = plt.gca().get_legend_handles_labels()

plt.subplot(1, 2, 2)
bins = np.linspace(0.0, 4.0, 50)
plt.hist(keras_response, bins=bins, histtype="step", density=True, label="Keras", lw=lw)
plt.hist(onnx_response, bins=bins, histtype="step", density=True, label="ONNX", linestyle="--", lw=lw)
plt.axvline(1.0, color="k", linestyle="--", lw=1, alpha=0.7)
plt.xlabel("Predicted / True Energy")
plt.ylabel("Density")
plt.xlim(0.0, 4.0)
plt.title("Response Distribution Comparison")
plt.text(0.95, 0.95, f"Keras Response RMSE: {keras_r_rmse:.4f}\nONNX Response RMSE: {onnx_r_rmse:.4f}", transform=plt.gca().transAxes, fontsize=10, verticalalignment="top", horizontalalignment="right")

fig.legend(handles, labels, loc="lower center", bbox_to_anchor=(0.5, -0.05), ncol=2)

plt.tight_layout()
plt.savefig(f"{PROJECT_ROOT}/data/results/{RUN}/export/onnx_comparison.png", dpi=300)
plt.savefig(f"{PROJECT_ROOT}/data/results/{RUN}/export/onnx_comparison.pdf")
plt.show()

In [ ]:
!cd {export_path} && \
tar -czvf benchmark_export_{RUN}.tgz \
    model.onnx \
    X_test.npy \
    keras_outputs.npz

In [ ]:
print(f"scp {export_path}/benchmark_export_{RUN}.tgz")